<a href="https://colab.research.google.com/github/RushiKP14/Tensorflow/blob/main/fcc_book_recommendation_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [134]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [135]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2025-03-07 15:37:42--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 172.67.70.149, 104.26.3.33, 104.26.2.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|172.67.70.149|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip.1’

book-crossings.zip. 100%[===================>]  24.88M  18.8MB/s    in 1.3s    

2025-03-07 15:37:44 (18.8 MB/s) - ‘book-crossings.zip.1’ saved [26085508/26085508]

Archive:  book-crossings.zip
replace BX-Book-Ratings.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [136]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:
# add your code here - consider creating a new cell for each section of code
df_books.head()
df_ratings.head()
#df_ratings.shape
#df_ratings.describe()

,user,isbn,rating
0,276725,034545104X,0.0
1,276726,0155061224,5.0
2,276727,0446520802,0.0
3,276729,052165615X,3.0
4,276729,0521795028,6.0


In [ ]:
import pandas as pd

# create a sample DataFrame
df = pd.DataFrame({
    'order_id': [1, 2, 3, 4],
    'customer_id': [101, 102, 103, 104],
    'product_name': ['Coca Cola', 'Pepsi', 'Fanta', 'Sprite'],
    'quantity': [2, 1, 3, 2]
})
print(df)
mask = df['product_name'] == 'Coca Cola'

# select all rows except the ones that contain 'Coca Cola'
df = df[~mask]

# print the resulting DataFrame
print(df)

   order_id  customer_id product_name  quantity
0         1          101    Coca Cola         2
1         2          102        Pepsi         1
2         3          103        Fanta         3
3         4          104       Sprite         2
   order_id  customer_id product_name  quantity
1         2          102        Pepsi         1
2         3          103        Fanta         3
3         4          104       Sprite         2
  product_name
1        Pepsi
2        Fanta
3       Sprite


In [137]:
list_user = df_ratings.user.value_counts()
print(list_user)
keep_user=[]
for i in list_user.index:
  if list_user.get(i)>=200:
    keep_user.append(i)
#print(keep_user)
flag=1
for user in keep_user:
  mask = df_ratings['user'] == user
  if flag==1:
    df_ratings_new = df_ratings.loc[mask]
    flag=0
  else:
    df_ratings_new = pd.concat([df_ratings_new, df_ratings.loc[mask]], ignore_index=True)
print(df_ratings_new)
sum(list_user.get(keep_user))

user
11676     13602
198711     7550
153662     6109
98391      5891
35859      5850
          ...  
116180        1
116166        1
116154        1
116137        1
276723        1
Name: count, Length: 105283, dtype: int64
         user            isbn  rating
0       11676      9022906116     7.0
1       11676  \0432534220\""     6.0
2       11676  \2842053052\""     7.0
3       11676   0 7336 1053 6     0.0
4       11676     0=965044153     7.0
...       ...             ...     ...
527551  26883      3401015834     7.0
527552  26883      3423000015     0.0
527553  26883      3442720117     7.0
527554  26883      3822505986     0.0
527555  26883      3929017245     0.0

[527556 rows x 3 columns]


527556

In [146]:
list_isbn = df_ratings.isbn.value_counts()
print(list_isbn)
keep_isbn=[]
for i in list_isbn.index:
  if list_isbn.get(i)>=100:
    keep_isbn.append(i)
print(keep_isbn)
flag=1
for isbn in keep_isbn:
  mask = df_ratings_new['isbn'] == isbn
  if flag==1:
    df_ratings_final = df_ratings_new.loc[mask]
    flag=0
  else:
    df_ratings_final = pd.concat([df_ratings_final, df_ratings_new.loc[mask]], ignore_index=True)
print(df_ratings_final)
#sum(list_isbn.get(keep_isbn))

isbn
0971880107     2502
0316666343     1295
0385504209      883
0060928336      732
0312195516      723
               ... 
1568656386        1
1568656408        1
1569551553        1
1570081808        1
05162443314       1
Name: count, Length: 340556, dtype: int64
['0971880107', '0316666343', '0385504209', '0060928336', '0312195516', '044023722X', '0679781587', '0142001740', '067976402X', '0671027360', '0446672211', '059035342X', '0316601950', '0375727345', '044021145X', '0452282152', '0440214041', '0804106304', '0440211727', '0345337662', '0060930535', '0440226430', '0312278586', '0743418174', '0671021001', '0345370775', '0446605239', '0156027321', '0440241073', '0671003755', '0060976845', '1400034779', '0786868716', '0440234743', '0440222656', '0440221471', '0345361792', '0440236673', '0345417623', '0316769487', '0385484518', '0446610038', '0446310786', '044022165X', '0375706771', '0440220602', '0440225701', '0060502258', '0446606812', '0345353145', '044651652X', '0140293248', '034

In [132]:
#ratings_neighb.loc[11676,'0971880107'] = df_ratings_final[(df_ratings_final['user']==11676) & (df_ratings_final['isbn']=='0971880107')]['rating']
#ratings_neighb.loc[11676,'0971880107']=df_ratings_final[(df_ratings_final['user']==11676) & (df_ratings_final['isbn']=='0971880107')]['rating'].values[0]
#ratings_neighb
df_ratings_final.loc[df_ratings_final['isbn']=='0971880107', 'rating']
data = np.ones((len(keep_user),len(keep_isbn)))*5
i,j = 0,0
data[i,j] = df_ratings_final[(df_ratings_final['user']==keep_user[i]) & (df_ratings_final['isbn']==keep_isbn[j])]['rating'].values[0]
type(df_ratings_final[(df_ratings_final['user']==keep_user[i]) & (df_ratings_final['isbn']==keep_isbn[j])]['rating'].ndim)
#print(data)
data.shape
data[905,730]

IndexError: index 905 is out of bounds for axis 0 with size 905

In [133]:
#df_ratings_final.loc[df_ratings_final['user']==11676]
data = np.ones((len(keep_user),len(keep_isbn)))*5
#ratings_neighb = pd.DataFrame(index = keep_user, columns = keep_isbn)
for i in range(len(keep_user)):
  for j in range(len(keep_isbn)):
    if df_ratings_final[(df_ratings_final['user']==keep_user[i]) & (df_ratings_final['isbn']==keep_isbn[j])]['rating'].empty:
      continue
    else:
      data[i,j] = df_ratings_final[(df_ratings_final['user']==keep_user[i]) & (df_ratings_final['isbn']==keep_isbn[j])]['rating'].values[0]
#ratings_neighb['0971880107'] = df_ratings_final.loc[df_ratings_final['isbn']=='0971880107', 'rating']

KeyboardInterrupt: 

In [175]:
data = df_ratings_final.pivot(index='isbn', columns='user', values='rating')
new_data = data.fillna(5.0)
print(new_data)
ratings_numpy = new_data.to_numpy()
print(ratings_numpy)

user        254     2276    2766    2977    3363    4017    4385    6242    \
isbn                                                                         
002542730X     5.0     5.0     5.0     5.0     0.0     5.0     5.0     5.0   
0060008032     5.0     5.0     5.0     5.0     5.0     5.0     5.0     5.0   
0060096195     5.0     5.0     5.0     5.0     0.0     5.0     5.0     5.0   
006016848X     5.0     5.0     5.0     5.0     5.0     5.0     5.0     5.0   
0060173289     5.0     5.0     5.0     5.0     5.0     5.0     5.0     5.0   
...            ...     ...     ...     ...     ...     ...     ...     ...   
1573227331     5.0     5.0     5.0     5.0     5.0     5.0     5.0     6.0   
1573229326     5.0     5.0     5.0     5.0     5.0     5.0     5.0     6.0   
1573229571     5.0     5.0     5.0     5.0     5.0     5.0     5.0     5.0   
1592400876     5.0     5.0     5.0     5.0     5.0     5.0     5.0     5.0   
1878424319     5.0     5.0     5.0     5.0     5.0     5.0     5

In [176]:
neigh = NearestNeighbors(n_neighbors=5).fit(ratings_numpy)

In [225]:
print(df_books.head())
new_data.loc['002542730X']
x = df_books.loc[df_books['title']=="Where the Heart Is (Oprah's Book Club (Paperback))",'isbn'].to_numpy()[0]
new_data.loc[x].to_numpy()
new_data.index[[0,1]]

         isbn                                              title  \
0  0195153448                                Classical Mythology   
1  0002005018                                       Clara Callan   
2  0060973129                               Decision in Normandy   
3  0374157065  Flu: The Story of the Great Influenza Pandemic...   
4  0393045218                             The Mummies of Urumchi   

                 author  
0    Mark P. O. Morford  
1  Richard Bruce Wright  
2          Carlo D'Este  
3      Gina Bari Kolata  
4       E. J. W. Barber  


Index(['002542730X', '0060008032'], dtype='object', name='isbn')

In [233]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):
  isbn = df_books[df_books['title']==book]['isbn'].to_numpy()[0]
  X = new_data.loc[isbn].to_numpy()
  distances, indices = neigh.kneighbors([X])
  recommended_isbns = new_data.index[indices[0]]
  recommended_books = [book]
  for i in range(5):
    recommended_books.append([df_books.loc[df_books['isbn']==recommended_isbns[i], 'title'].to_numpy()[0], distances[i]])
  return recommended_books

In [230]:
book="Where the Heart Is (Oprah's Book Club (Paperback))"
isbn = df_books[df_books['title']==book]['isbn'].to_numpy()[0]
print(isbn)

0446672211


In [234]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")

IndexError: index 1 is out of bounds for axis 0 with size 1

In [222]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

InvalidIndexError: (0         False
1         False
2         False
3         False
4         False
          ...  
271374    False
271375    False
271376    False
271377    False
271378    False
Name: title, Length: 271379, dtype: bool, 'isbn')